# Cleaning4
Livello di pulizia che prende i file cleaned_01 fa le elaborazioni di cleaning 2 & 3 ma senza eliminare righe.\
File così creati servono per il merge.
## Inizializzazione ed Import

In [1]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from data_model.manage_excel_support_file import *
import pandas as pd
import os

client = DatalakeClient()

# Download the raw files 
Exclusevily from ADNI dataset stored in the Datalake

In [2]:
file_codes = ['ADNIMERGE', 'ADSP_PHC_BIOMARKER', 'ADNI_DIAN_COMPARISON', 'MMSE', 'ADAS', 'FAQ', 'CDR', 'MOCA']


#'ADNIMERGE', 'PTDEMOG', 'DXSUM', 'MMSE', 'ADAS', 'FAQ', 'CDR', 'MOCA', 'APOERES', 
#            'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSX51_ADNI1_3T', 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSL', 'UCSDVOL', 'UPENN_ROI_MARS'

### Mixed info ###
# 'ADNIMERGE', 
# 'ADSP_PHC_BIOMARKER', 'ADNI_DIAN_COMPARISON', --> have CSF

### Single Cofactor ###
# 'PTDEMOG', 'DXSUM', 'MMSE', 'ADAS', 'FAQ', 'CDR', 'MOCA', 'APOERES'

### Volumes ###
# 'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSL', 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSX51_ADNI1_3T'
# 'UCSDVOL', 'UPENN_ROI_MARS',  --> do NOT use FreeSurfer but other model or Atlas so not comparable

### CSF ###
# 'UPENNBIOMK_ADNIDIAN_ES_2017', 'UPENNBIOMK_ROCHE_ELECSYS', 'EUROIMMUN', 'FUJIREBIOABETA', 'SALADAX_BIOMEDICAL', 'MESOSCALE', 'UPENNBIOMK_MASTER', 'UPENN_2DUPLC_CRM', 

In [3]:
search = client.query_files(
    query={'custom.level' : 'cleaned_01', 'custom.source' : 'ADNI', 'custom.file_code': file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)


In [4]:
len(zip_files)

8

## Operazioni
- Trasformare in dummies alcuni parametri
- Normalizzare i volumi
- nuovi metadati (cofattori e fattori)

In [5]:
# create new support file with the info from the new dataset after cleaning 2
support_file_path = 'ADNI_variables_cleaned1'
support_file = pd.read_excel(support_file_path+'.xlsx')
new_name = 'ADNI_variables_cleaned4'

if os.path.isfile(new_name+'.xlsx'):
    #aggiunge i filecode mancanti e riporta i file_code da riprocessare allo status precedente (variable names)
    update_new_support_file(support_file, new_name, processed_file=file_codes)
else:
    create_new_support_file(support_file, support_file_path, new_name=new_name, rename_column=False)


The ADNI_variables_cleaned4 file has been updated with the new file_code: []
Open the file and verify it, if needed update the variables names and metadata
The ADNI_variables_cleaned4 file has restored the previous information of the file_code: ['ADNIMERGE', 'ADSP_PHC_BIOMARKER', 'ADNI_DIAN_COMPARISON', 'MMSE', 'ADAS', 'FAQ', 'CDR', 'MOCA']
Open the file and verify it, if needed update the variables names and metadata


In [6]:
new_support_file = pd.read_excel(new_name+'.xlsx')
dataCleaner = DataCleaner(support_file=new_support_file)

In [7]:
for file_name in zip_files.keys():
    print('\n\n ----', file_name)
    
    df = zip_files[file_name]
    df_new = df.copy(deep=True)
    new_support_file = dataCleaner.update_self_support_file(new_support_file)
    
    file_code, metadat_custom = dataCleaner.get_file_code_metadata(file_name, prefix='cleaned/single_file')

    processed_df = df_new.copy(deep=True)

    # funzione che trasforma parametri categorici in dummies
    ref_list = ['GENDER', 'MARRY', 'ETHNICITY', 'RACE', 'DX']
    to_dummy_list = [x for x in ref_list if x in processed_df.columns]
    if to_dummy_list:
        print('\n\n Dummies in ----', file_name)
        processed_df, bool_var = dataCleaner.classes_to_dummies(processed_df, col_list=to_dummy_list) 
    
    if 'volume' in support_file[support_file['file_code'] == file_code]['metadati_normalizzazione'].values:
        print('\n\n Volumes in ----', file_name)
        # verifica che i volumi siano già totali e non solo una parte laterale
        processed_df, volume_list = dataCleaner.get_volumes_total(processed_df, file_code)
        print(processed_df.columns)
        # Transform volumes as ICV percentage
        processed_df = dataCleaner.transform_volumes_as_ICV_percent(processed_df, volume_list, file_code)
    
    final_df = processed_df.copy(deep=True)
    
    if 'FSVERSION' in final_df.columns:
        print(final_df['FSVERSION'].unique())

    new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_04')

    # aggiunta di righe per i nuovi parametri e rimozione dal support di variabili non più nel df
    new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
    
    # Ottenimento dei metadati dal support file (cofattori/fattori, scal/intervallo)
    updated_metadata = dataCleaner.extract_metadata_from_support(final_df, new_level='cleaned_04', file_name=file_name, prefix='cleaned/single_file', updated_support_file=new_support_file)    
    
    # get info into the new support file
    infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name, prefix='cleaned/single_file')
    
    for key in final_df.keys():
        if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
            new_support_file = infoSupportFile.get_varible_info(key)
            new_support_file = infoSupportFile.get_subjects_and_multiplevisits(key)
    
    # upload the new file
    result = client.upload_dataframe(
        df=final_df,
        object_name=new_file_name,
        prefix='cleaned/single_file/',
        metadata=updated_metadata
    )

save_df(df_to_save=new_support_file, output_path=new_name) 



 ---- ADNIMERGE_25Jul2025_01.csv


 Dummies in ---- ADNIMERGE_25Jul2025_01.csv


 Volumes in ---- ADNIMERGE_25Jul2025_01.csv
Index(['RID', 'COHORT', 'VISCODE', 'VISIT_MONTH', 'EXAMDATE', 'AGE_bl',
       'EDUCAT', 'APOE_4', 'CDRSB', 'ADAS11', 'ADAS13', 'MMSE',
       'RAVLT_immediate', 'FAQ', 'MOCA', 'FLDSTRENG', 'FSVERSION', 'IMAGEUID',
       'Ventricles', 'Hippocampus', 'Entorhinal', 'Fusiform', 'MidTemp', 'ICV',
       'update_stamp', 'AGE', 'GENDER/female', 'GENDER/male', 'MARRY/divorced',
       'MARRY/married', 'MARRY/single', 'MARRY/widowed', 'ETHNICITY/latino',
       'ETHNICITY/not_latino', 'RACE/Asian', 'RACE/Black', 'RACE/Mixed',
       'RACE/Native_american', 'RACE/White', 'DX/CN', 'DX/Dementia', 'DX/MCI'],
      dtype='object')
[4.3 nan 5.1 6. ]


 ---- MMSE_25Jul2025_01.csv


 ---- ADSP_PHC_BIOMARKER_25Jul2025_01.csv


 Dummies in ---- ADSP_PHC_BIOMARKER_25Jul2025_01.csv


 ---- ADNI-DIAN_Comparison_Study_Data_Subset_05_23_22_23Oct2025_01.csv


 Dummies in ---- ADNI-DI

In [8]:
final_df

,RID,VISCODE,VISIT_MONTH,EXAMDATE,FLDSTRENG,IMAGEUID,STATUS,FSVERSION,ICV%ICV,Entorhinal%ICV,Fusiform%ICV,Hippocampus%ICV,Ventricles%ICV,MidTemp%ICV
0,3,sc,0,2005-09-01,1.5T,32237,complete,4.4,100.0,0.122404,0.845425,0.270528,3.737928,0.939454
1,3,m06,6,2006-03-13,1.5T,31863,complete,4.4,100.0,0.123634,0.788752,0.261011,3.931380,0.929958
2,3,m12,12,2006-09-12,1.5T,35576,complete,4.4,100.0,0.113981,0.766722,0.263418,3.993340,0.912534
3,3,m24,24,2007-09-12,1.5T,88252,complete,4.4,100.0,0.105704,0.721753,0.251022,4.330573,0.856931
4,4,sc,0,2005-09-22,1.5T,64631,complete,4.4,100.0,0.231208,1.160506,0.380186,2.289870,1.163662
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3382,1427,m18,19,2009-03-12,1.5T,143375,complete,4.4,100.0,0.222350,1.021851,0.471934,1.398308,1.391516
3383,1427,m24,24,2009-08-26,1.5T,163861,complete,4.4,100.0,0.226921,1.064123,0.471741,1.304354,1.455481
3384,1427,m48,48,2011-08-19,1.5T,274711,complete,4.4,100.0,0.214258,1.056177,0.473785,1.503969,1.470185
3385,1430,sc,0,2007-09-07,1.5T,79857,complete,4.4,100.0,0.203600,1.073505,0.311156,1.861098,1.087813


In [9]:
updated_metadata

{'file_code': 'UCSFFSL',
 'level': 'cleaned_04',
 'population': ['ADNI1', 'ADNIGO', 'ADNI2'],
 'source': 'ADNI',
 'cofattori': [],
 'predittori': ['ICV%ICV',
  'Entorhinal%ICV',
  'Fusiform%ICV',
  'Hippocampus%ICV',
  'Ventricles%ICV',
  'MidTemp%ICV'],
 'norm_scala': [],
 'norm_intervallo': [],
 'norm_volume': ['ICV%ICV',
  'Entorhinal%ICV',
  'Fusiform%ICV',
  'Hippocampus%ICV',
  'Ventricles%ICV',
  'MidTemp%ICV'],
 'norm_scale_value': [],
 'volume_norm_values': {'Ventricles%ICV': [0.11, 7.3, 'increasing'],
  'Hippocampus%ICV': [0.22, 0.66, 'inverse'],
  'Entorhinal%ICV': [0.07, 0.39, 'inverse'],
  'Fusiform%ICV': [0.63, 1.62, 'inverse'],
  'MidTemp%ICV': [0.7, 1.8, 'inverse']}}